# 🔬 Model Explainability (SHAP & LIME)
**Extended · Data Pattern Recognition for the Rest of Us**

> A black-box model that performs well is only half the job. Regulators, customers, and business leaders need to understand *why* a prediction was made. SHAP and LIME make any model explainable.

### Dataset
**Loan Approval / Credit Risk**
Predict loan default using applicant features: income, employment, credit history, debt ratio. This is one of the highest-stakes explainability domains — regulations (ECOA, FCRA) often require lenders to provide specific reasons for adverse decisions.

### Time: ~65 minutes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa',
    'axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
!pip install shap -q
import shap
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.preprocessing import StandardScaler
print("\u2713 Setup complete — shap installed")

In [ ]:
# Loan Application Dataset
# Predict probability of default — a regulated, high-stakes prediction
np.random.seed(42)
n = 5000

# Applicant features
credit_score     = np.random.randint(300, 850, n)
annual_income    = np.random.lognormal(11, 0.5, n).astype(int).clip(15000, 500000)
employment_years = np.random.randint(0, 30, n).astype(float)
debt_to_income   = np.random.beta(2, 5, n).clip(0.01, 0.99)
loan_amount      = np.random.lognormal(10.5, 0.5, n).astype(int).clip(1000, 100000)
num_credit_lines = np.random.randint(1, 20, n)
late_payments    = np.random.choice([0,1,2,3,4,5], n, p=[0.60,0.18,0.10,0.06,0.04,0.02])
home_ownership   = np.random.choice(['Rent','Own','Mortgage'], n, p=[0.35,0.20,0.45])
loan_purpose     = np.random.choice(['Debt Consolidation','Home Improvement',
                                     'Auto','Business','Education'], n,
                                    p=[0.40,0.20,0.15,0.15,0.10])

# Default probability — driven by credit score, DTI, late payments
log_odds = (-1
            - 0.008 * (credit_score - 600)
            + 2.5   * debt_to_income
            + 0.4   * late_payments
            - 0.0000015 * annual_income
            - 0.03  * employment_years
            + 0.000003 * loan_amount)
prob_default = 1 / (1 + np.exp(-log_odds))
default = (np.random.uniform(0,1,n) < prob_default).astype(int)

df = pd.DataFrame({
    'CreditScore': credit_score, 'AnnualIncome': annual_income,
    'EmploymentYears': employment_years, 'DebtToIncome': debt_to_income,
    'LoanAmount': loan_amount, 'NumCreditLines': num_credit_lines,
    'LatePayments': late_payments,
    'HomeOwnership': pd.Categorical(home_ownership).codes,
    'LoanPurpose': pd.Categorical(loan_purpose).codes,
    'Default': default
})

feature_cols = [c for c in df.columns if c != 'Default']
X = df[feature_cols]
y = df['Default']
print(f"Loan dataset: {n:,} applications")
print(f"Default rate: {y.mean():.1%}")
print(f"\nFeatures: {feature_cols}")

## 📐 Part 1 — SHAP Values: Explaining Every Prediction

**SHAP** (SHapley Additive exPlanations) assigns each feature a contribution to the difference between this specific prediction and the average prediction.

```
prediction = base_value + SHAP(CreditScore) + SHAP(DTI) + SHAP(LatePayments) + ...
```

Key properties:
- **Efficiency:** SHAP values sum exactly to the prediction deviation from baseline
- **Consistency:** features that contribute more always get higher SHAP values
- **Local + Global:** explain one prediction OR summarize across all predictions

In [ ]:
# Fit a GBM model on loan data
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25,
                                             random_state=42, stratify=y)
gbm = GradientBoostingClassifier(n_estimators=200, max_depth=4,
                                  learning_rate=0.05, random_state=42)
gbm.fit(X_tr, y_tr)
print(f"GBM AUC-ROC: {roc_auc_score(y_te, gbm.predict_proba(X_te)[:,1]):.4f}")
print("\nCalculating SHAP values (may take ~30 seconds)...")

explainer = shap.TreeExplainer(gbm)
shap_values = explainer.shap_values(X_te)
print("\u2713 SHAP values computed")

In [ ]:
# Global: SHAP summary plot — what drives default risk overall?
plt.figure(figsize=(9, 6))
shap.summary_plot(shap_values, X_te, plot_type='bar', show=False,
                  plot_size=(9, 5))
plt.title('Global Feature Importance — What Drives Loan Default Risk?\n'
          'Average absolute SHAP value across all predictions')
plt.tight_layout(); plt.show()

In [ ]:
# SHAP beeswarm — direction AND magnitude
plt.figure(figsize=(9, 6))
shap.summary_plot(shap_values, X_te, show=False, plot_size=(9, 6))
plt.title('SHAP Beeswarm Plot — Direction and Magnitude\n'
          'Red = high feature value, Blue = low feature value')
plt.tight_layout(); plt.show()
print("\n\u2714 Reading this plot:")
print("  CreditScore: high score (red) = negative SHAP = reduces default risk")
print("  DebtToIncome: high DTI (red) = positive SHAP = increases default risk")
print("  LatePayments: any late payments push default risk up significantly")

In [ ]:
# @title 🔍 Local Explanation — Select an Applicant Profile
# @markdown Choose an applicant below and click ▶ Run to see their SHAP waterfall explanation.
# @markdown Each profile represents a realistic loan scenario — compare how the model reasons differently across risk levels.

APPLICANT_PROFILE = "Borderline — medium risk" # @param ["High Risk — likely decline", "Borderline — medium risk", "Low Risk — likely approve", "Young professional — thin credit file", "High income — high debt", "Senior applicant — long history"]

# ── Pre-defined applicant profiles ───────────────────────────
# Each is a realistic loan scenario designed to highlight different SHAP patterns
PROFILES = {
    "High Risk — likely decline": {
        "CreditScore":      520,
        "AnnualIncome":     32000,
        "EmploymentYears":  1,
        "DebtToIncome":     0.72,
        "LoanAmount":       18000,
        "NumCreditLines":   3,
        "LatePayments":     4,
        "HomeOwnership":    0,   # Rent
        "LoanPurpose":      0,   # Debt Consolidation
        "description":      "Recent college graduate, unstable employment, multiple late payments, high DTI"
    },
    "Borderline — medium risk": {
        "CreditScore":      648,
        "AnnualIncome":     58000,
        "EmploymentYears":  4,
        "DebtToIncome":     0.41,
        "LoanAmount":       12000,
        "NumCreditLines":   7,
        "LatePayments":     1,
        "HomeOwnership":    2,   # Mortgage
        "LoanPurpose":      1,   # Home Improvement
        "description":      "Mid-career professional, one past late payment, moderate debt load"
    },
    "Low Risk — likely approve": {
        "CreditScore":      780,
        "AnnualIncome":     105000,
        "EmploymentYears":  12,
        "DebtToIncome":     0.18,
        "LoanAmount":       8000,
        "NumCreditLines":   11,
        "LatePayments":     0,
        "HomeOwnership":    1,   # Own
        "LoanPurpose":      2,   # Auto
        "description":      "Established professional, homeowner, excellent credit, low debt"
    },
    "Young professional — thin credit file": {
        "CreditScore":      690,
        "AnnualIncome":     72000,
        "EmploymentYears":  2,
        "DebtToIncome":     0.28,
        "LoanAmount":       6000,
        "NumCreditLines":   2,
        "LatePayments":     0,
        "HomeOwnership":    0,   # Rent
        "LoanPurpose":      4,   # Education
        "description":      "Good income and no late payments, but short credit history — thin file risk"
    },
    "High income — high debt": {
        "CreditScore":      710,
        "AnnualIncome":     195000,
        "EmploymentYears":  8,
        "DebtToIncome":     0.65,
        "LoanAmount":       45000,
        "NumCreditLines":   14,
        "LatePayments":     0,
        "HomeOwnership":    2,   # Mortgage
        "LoanPurpose":      3,   # Business
        "description":      "High earner but carrying significant debt — does income offset the DTI?"
    },
    "Senior applicant — long history": {
        "CreditScore":      755,
        "AnnualIncome":     68000,
        "EmploymentYears":  30,
        "DebtToIncome":     0.22,
        "LoanAmount":       10000,
        "NumCreditLines":   16,
        "LatePayments":     0,
        "HomeOwnership":    1,   # Own
        "LoanPurpose":      0,   # Debt Consolidation
        "description":      "Near retirement, long stable history, owns home outright"
    },
}

# ── Run explanation for selected profile ─────────────────────
profile_data = PROFILES[APPLICANT_PROFILE]
desc = profile_data.pop("description", "")

applicant = pd.DataFrame([{k: v for k, v in profile_data.items()}])[feature_cols]
profile_data["description"] = desc  # put it back

pred_prob          = gbm.predict_proba(applicant)[0, 1]
shap_for_applicant = explainer.shap_values(applicant)[0]
decision           = "DECLINE" if pred_prob > 0.5 else "APPROVE"
decision_color     = "\033[91m" if decision == "DECLINE" else "\033[92m"
R                  = "\033[0m"

print(f"\n{'='*58}")
print(f"  LOAN APPLICATION REVIEW")
print(f"{'='*58}")
print(f"  Profile:  {APPLICANT_PROFILE}")
print(f"  Context:  {desc}")
print(f"{'─'*58}")
print(f"  {'Feature':<22} {'Value':>12}")
print(f"{'─'*58}")
display_names = {
    "CreditScore": "Credit Score",
    "AnnualIncome": "Annual Income ($)",
    "EmploymentYears": "Employment Years",
    "DebtToIncome": "Debt-to-Income Ratio",
    "LoanAmount": "Loan Amount ($)",
    "NumCreditLines": "# Credit Lines",
    "LatePayments": "Late Payments",
    "HomeOwnership": "Home Ownership",
    "LoanPurpose": "Loan Purpose",
}
home_labels = {0:"Rent", 1:"Own", 2:"Mortgage"}
purpose_labels = {0:"Debt Consolidation", 1:"Home Improvement",
                  2:"Auto", 3:"Business", 4:"Education"}

for feat in feature_cols:
    val = applicant[feat].values[0]
    if feat == "HomeOwnership":
        display_val = home_labels.get(int(val), str(val))
    elif feat == "LoanPurpose":
        display_val = purpose_labels.get(int(val), str(val))
    elif feat in ("AnnualIncome", "LoanAmount"):
        display_val = f"${val:,.0f}"
    elif feat == "DebtToIncome":
        display_val = f"{val:.0%}"
    else:
        display_val = f"{val:.0f}"
    print(f"  {display_names.get(feat, feat):<22} {display_val:>12}")

print(f"{'─'*58}")
print(f"  Default Probability: {pred_prob:.1%}")
print(f"  Decision:            {decision_color}{decision}{R}")
print(f"{'='*58}\n")

# SHAP waterfall
shap_df = pd.DataFrame({
    "Feature": feature_cols,
    "SHAP":    shap_for_applicant,
    "Value":   applicant.values[0]
}).sort_values("SHAP", ascending=False)

print("Top reasons driving this decision:")
for _, row in shap_df.head(4).iterrows():
    arrow = "\u2191 INCREASES" if row["SHAP"] > 0 else "\u2193 REDUCES"
    risk  = "default risk" if row["SHAP"] > 0 else "default risk"
    print(f"  {row['Feature']:<22} {arrow} {risk}  (SHAP={row['SHAP']:+.3f})")

print()
fig, ax = plt.subplots(figsize=(11, 5))
shap.waterfall_plot(shap.Explanation(
    values=shap_for_applicant,
    base_values=explainer.expected_value,
    data=applicant.values[0],
    feature_names=feature_cols), show=False)
plt.title(f"{APPLICANT_PROFILE}\n"
          f"Decision: {decision}  |  Default probability: {pred_prob:.1%}",
          fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# Adverse action notice — plain English for the applicant
# (Required by ECOA/FCRA for any credit denial)

top_adverse = shap_df[shap_df["SHAP"] > 0].head(3)
top_positive= shap_df[shap_df["SHAP"] < 0].head(2)

print("=" * 60)
print(f"ADVERSE ACTION NOTICE" if decision == "DECLINE" else "APPROVAL SUMMARY")
print("=" * 60)
print(f"Application decision: {decision}")
print(f"Estimated default probability: {pred_prob:.0%}\n")

if decision == "DECLINE":
    print("Primary reasons for this decision:")
    for i, (_, row) in enumerate(top_adverse.iterrows(), 1):
        name = row["Feature"].replace("_", " ").replace("Num", "Number of")
        print(f"  {i}. {name} (contribution score: {row['SHAP']:+.3f})")
    if len(top_positive):
        print("\nFactors working in the applicant's favour:")
        for _, row in top_positive.iterrows():
            name = row["Feature"].replace("_", " ").replace("Num", "Number of")
            print(f"  + {name} (contribution score: {row['SHAP']:+.3f})")
    print("\nNote: Improving the top adverse factors above would")
    print("most improve this applicant's credit profile.")
else:
    print("This application meets our credit criteria.")
    if len(top_adverse):
        print("\nNote: The following factors were monitored:")
        for _, row in top_adverse.iterrows():
            name = row["Feature"].replace("_", " ").replace("Num", "Number of")
            print(f"  - {name} (contribution: {row['SHAP']:+.3f})")

print("=" * 60)
print("\n\U0001f4a1 Try a different profile using the dropdown above")
print("   Compare how the SHAP waterfall changes across applicants")


In [ ]:
# @title 📝 Quiz — Model Explainability (SHAP & LIME)
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What does a SHAP value of +0.25 for DebtToIncome mean?
# @markdown **Q2:** How is SHAP different from standard feature importance (like Gini importance)?
# @markdown **Q3:** What regulatory requirement makes explainability essential for loan decisions?
# @markdown **Q4:** What is LIME and how does it differ from SHAP?
# @markdown **Q5:** Write a 2-sentence plain English explanation of a loan decline for a non-technical applicant.

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

# Collect into answers dict for grading cell
answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the AI grading cell below for feedback")

### 📤 Submit for AI Grading

In [ ]:
_NB_NAME_ = "Model Explainability (SHAP & LIME)"
# @title 🤖 AI Feedback — enter your username and click ▶ Run
# @markdown **No API key needed.** AI grading runs free inside your Colab session.
# @markdown
# @markdown Enter your GitHub username, then click ▶ Run for question-by-question feedback.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"e.g. jsmith42"}

# ── runs automatically — do not edit below ───────────────────
import json, textwrap, re as _re, time
_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_quiz_questions():
    """Pull question text from the quiz cell @markdown lines."""
    try:
        import ipynbname
        nb_path = ipynbname.path()
        with open(nb_path) as f:
            nb = json.load(f)
        for cell in nb["cells"]:
            src = "".join(cell.get("source", []))
            if "@title" in src and "Quiz" in src:
                qs = re.findall(r"@markdown \*\*Q\d+:\*\* (.+)", src)
                if qs: return qs
    except Exception:
        pass
    return []

def _call_gemini(prompt, max_retries=3):
    """Call Gemini with retry on 429 rate limit."""
    last_err = None
    for attempt in range(max_retries):
        try:
            import google.generativeai as genai
            import google.auth, google.auth.transport.requests
            creds, _ = google.auth.default()
            creds.refresh(google.auth.transport.requests.Request())
            genai.configure(credentials=creds)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(
                prompt,
                generation_config={"max_output_tokens": 1500, "temperature": 0.3}
            )
            return result.text, "Gemini via Colab"
        except Exception as e:
            last_err = str(e)
            if "429" in str(e) or "quota" in str(e).lower():
                wait = 2 ** attempt
                print(f"  Rate limit hit — waiting {wait}s before retry {attempt+1}/{max_retries}...")
                time.sleep(wait)
                continue
            break
    # Try stored key as fallback
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            import google.generativeai as genai
            genai.configure(api_key=key)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(prompt)
            return result.text, "Gemini via key"
    except Exception:
        pass
    return None, last_err

def _build_prompt(answers_dict, nb_name, questions):
    answered  = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total   = len(answers_dict)
    n_done    = len(answered)

    # Pair each question with the student answer
    qa_pairs = ""
    for i, (k, v) in enumerate(answers_dict.items(), 1):
        q_text = questions[i-1] if i-1 < len(questions) else f"Question {i}"
        a_text = str(v).strip() if str(v).strip() else "(no answer)"
        qa_pairs += f"Q{i}: {q_text}\nA{i}: {a_text}\n\n"

    return f"""You are a TA grading a student quiz for the "{nb_name}" notebook in a data science course called "Data Pattern Recognition for the Rest of Us" (based on ISLP and business analytics).

{qa_pairs.strip()}

For EACH question:
- Decide if the answer is CORRECT, PARTIALLY CORRECT, or INCORRECT
- A short paraphrase or reasonable approximation counts as CORRECT — do not require exact wording
- For INCORRECT or PARTIAL: name the specific concept they need to review (1 sentence)
- For CORRECT: brief confirmation of what they got right (1 sentence)

Then give an overall summary.

Respond ONLY in this exact JSON format (no markdown fences, no extra text):
{{
  "questions": [
    {{
      "q": 1,
      "status": "<CORRECT|PARTIAL|INCORRECT>",
      "comment": "<one specific sentence>"
    }}
  ],
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent|Good|Needs Review|Incomplete>",
  "summary": "<2 sentences overall: strengths and what to revisit>",
  "review_tip": "<one specific concept, chapter, or notebook to review if they struggled — or 'Great work!' if excellent>"
}}

Scoring guide: CORRECT=1pt, PARTIAL=0.5pt (round to nearest int), INCORRECT=0pt."""

def _show(result, username, nb_name, source, questions):
    STATUS_ICON  = {"CORRECT": "\u2705", "PARTIAL": "\u26a0\ufe0f", "INCORRECT": "\u274c"}
    STATUS_COLOR = {"CORRECT": "\033[92m", "PARTIAL": "\033[93m", "INCORRECT": "\033[91m"}
    R = "\033[0m"
    grade = result.get("grade", "?")
    GRADE_COLOR = {"Excellent":"\033[92m","Good":"\033[94m",
                   "Needs Review":"\033[93m","Incomplete":"\033[91m"}
    GC = GRADE_COLOR.get(grade, "")
    n  = len(answers)
    qs = result.get("quiz_score", 0)
    bar = "\u2588"*qs + "\u2591"*(n-qs)

    print("\n" + "\u2550"*60)
    print(f"  \U0001f916 AI Feedback \u2014 {nb_name}")
    if source: print(f"  Powered by   {source}")
    print("\u2550"*60)
    print(f"  Student  : {'@'+username if username else '\u26a0 set GITHUB_USERNAME above'}")
    print(f"  Grade    : {GC}{grade}{R}")
    print(f"  Score    : {qs}/{n}  [{bar}]")
    print()

    # Question-by-question breakdown
    q_results = result.get("questions", [])
    if q_results:
        print("  \u2500"*29)
        for qr in q_results:
            idx    = qr.get("q", 0) - 1
            status = qr.get("status", "?")
            icon   = STATUS_ICON.get(status, "\u2753")
            color  = STATUS_COLOR.get(status, "")
            comment= qr.get("comment", "")
            q_text = questions[idx] if idx < len(questions) else f"Question {qr.get('q','?')}"
            # Truncate long question text for display
            q_short = q_text[:55] + "..." if len(q_text) > 55 else q_text
            print(f"  {icon} {color}Q{qr.get('q','?')} {status}{R}")
            print(f"     {q_short}")
            if comment:
                for line in textwrap.wrap(comment, 54):
                    print(f"     \u2192 {line}")
            print()

    # Summary
    summary = result.get("summary", "")
    if summary:
        print("  \u2500"*29)
        print("  Overall:")
        for line in textwrap.wrap(summary, 56):
            print(f"  {line}")

    # Review tip
    tip = result.get("review_tip", "")
    if tip and tip != "Great work!":
        print()
        for line in textwrap.wrap(f"\U0001f4d6 Review: {tip}", 56):
            print(f"  {line}")
    elif tip == "Great work!":
        print()
        print("  \U0001f4d6 Great work! Keep going.")

    print("\u2550"*60 + "\n")

def _fallback_grade(answers_dict):
    """Smart fallback — grade on completion only, no length penalty."""
    n  = len(answers_dict)
    nd = sum(1 for v in answers_dict.values() if str(v).strip())
    if nd == 0:
        return {"quiz_score":0,"grade":"Incomplete",
                "questions":[],
                "summary":"No answers provided — fill in the quiz above.",
                "review_tip":"Complete the quiz and re-run for AI feedback."}
    elif nd == n:
        return {"quiz_score":n,"grade":"Good",
                "questions":[],
                "summary":f"All {n} questions answered. AI grading was unavailable — re-run to get question-by-question feedback.",
                "review_tip":"Re-run this cell in a few minutes for detailed AI feedback."}
    else:
        return {"quiz_score":nd,"grade":"Needs Review",
                "questions":[],
                "summary":f"{nd}/{n} questions answered. Complete the remaining {n-nd} and re-run.",
                "review_tip":"Answer all questions for full feedback."}

# ── Main ──────────────────────────────────────────────────────
if "answers" not in globals():
    print("\u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    nd       = sum(1 for v in answers.values() if str(v).strip())
    username = GITHUB_USERNAME.strip()
    questions = _get_quiz_questions()

    print(f"\n  Notebook : {_NB_TITLE}  \u2022  {nd}/{len(answers)} answered")
    if username: print(f"  Student  : @{username}")
    print(f"  Grading  : please wait...\n")

    prompt     = _build_prompt(answers, _NB_TITLE, questions)
    raw, src   = _call_gemini(prompt)

    if raw:
        try:
            clean  = _re.sub(r"```(?:json)?\s*|```","",raw).strip()
            result = json.loads(clean)
        except Exception:
            nd2    = sum(1 for v in answers.values() if str(v).strip())
            result = {"quiz_score":nd2,"grade":"Good" if nd2==len(answers) else "Needs Review",
                      "questions":[],"summary":raw[:400],"review_tip":""}
    else:
        if src: print(f"  \u26a0 Gemini unavailable ({src[:60]}) \u2014 showing completion feedback\n")
        result = _fallback_grade(answers)

    _show(result, username, _NB_TITLE, src if raw else None, questions)
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 your fork\n")


---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*